# Rapport de tests de résistance

Ce rapport lit `data/08_stress/`. Il ne réestime pas les poids. La page est `reports/stress/dashboard.html`. Elle est réécrite par `build_stress` à chaque reconstruction. La méthode est `documentation/09_stress_testing.ipynb`.

Le rendement est mensuel, simple, en francs. Un chiffre négatif est une perte. Les quatre chocs sont les paramètres du module. La cellule les imprime, puis elle vérifie que le fichier les retrouve.


In [1]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import (
    MARKET_SHOCK_RETURN,
    RATE_SHOCK_ANNUAL_POINTS,
    RECESSION_SHOCK_MONTHS,
    RECESSION_SHOCK_RETURN,
    VOLATILITY_SHOCK_MULTIPLIER,
)
from src.stress_testing.build_stress import AGREEMENT, PATHS_FILE, RESULTS_FILE

LABELS = {
    "equal_weight": "Équipondéré",
    "minimum_volatility": "Volatilité minimale",
    "maximum_sharpe": "Sharpe maximal",
}
SHOCKS = {
    "aucun": "Sans choc",
    "marche": "Marché",
    "taux": "Taux",
    "recession": "Récession",
    "volatilite": "Volatilité",
}


def percent(value):
    return f"{float(value) * 100:.1f}".replace(".", ",") + " %"


results = pd.read_csv(RESULTS_FILE)
paths = pd.read_csv(PATHS_FILE)
print(
    f"Marché : dernier mois remplacé par {percent(MARKET_SHOCK_RETURN)}. "
    f"Taux : SARON annuel décalé de {RATE_SHOCK_ANNUAL_POINTS:.0f} points. "
    f"Récession : {RECESSION_SHOCK_MONTHS} mois à {percent(RECESSION_SHOCK_RETURN)}. "
    f"Volatilité : écarts multipliés par {VOLATILITY_SHOCK_MULTIPLIER:.1f}."
)
start = results["window_start"].iloc[0]
end = results["window_end"].iloc[0]
print(f"Période {start} – {end}, {int(results['months'].iloc[0])} mois.")
for strategy, label in LABELS.items():
    print(label)
    block = results.loc[results["strategy"].eq(strategy)].set_index("shock")
    base = block.loc["aucun"]
    for shock, shock_label in SHOCKS.items():
        row = block.loc[shock]
        print(
            f"  {shock_label} : cumulée {float(row['terminal_wealth']):.2f}, "
            f"drawdown {percent(row['maximum_drawdown'])}, "
            f"volatilité {percent(row['annualized_volatility'])}, "
            f"Sharpe {float(row['sharpe_ratio']):.2f}."
        )
    if abs(float(block.loc["taux", "terminal_wealth"]) - float(base["terminal_wealth"])) > AGREEMENT:
        raise RuntimeError(f"{strategy} rate shock changed terminal wealth.")
    ratio = float(block.loc["volatilite", "annualized_volatility"]) / float(base["annualized_volatility"])
    if abs(ratio - VOLATILITY_SHOCK_MULTIPLIER) > AGREEMENT:
        raise RuntimeError(f"{strategy} volatility shock did not scale.")
    trail = paths.loc[paths["strategy"].eq(strategy)].sort_values("month")
    market = trail.loc[trail["shock"].eq("marche"), "portfolio_return"].iloc[-1]
    if abs(float(market) - MARKET_SHOCK_RETURN) > AGREEMENT:
        raise RuntimeError(f"{strategy} market shock did not replace the last month.")
    recession = trail.loc[trail["shock"].eq("recession"), "portfolio_return"].iloc[-RECESSION_SHOCK_MONTHS :]
    if float(np.max(np.abs(recession.to_numpy(dtype=float) - RECESSION_SHOCK_RETURN))) > AGREEMENT:
        raise RuntimeError(f"{strategy} recession shock did not replace the last months.")
print("Les trois livres retrouvent les paramètres. La récession impose le même rendement. Elle ne désigne pas un titre.")


Marché : dernier mois remplacé par -20,0 %. Taux : SARON annuel décalé de 2 points. Récession : 6 mois à -3,0 %. Volatilité : écarts multipliés par 1.5.
Période 2016-11 – 2026-10, 120 mois.
Équipondéré
  Sans choc : cumulée 2.15, drawdown -22,9 %, volatilité 12,3 %, Sharpe 0.69.
  Marché : cumulée 1.68, drawdown -22,9 %, volatilité 13,9 %, Sharpe 0.45.
  Taux : cumulée 2.15, drawdown -22,9 %, volatilité 12,3 %, Sharpe 0.53.
  Récession : cumulée 1.64, drawdown -23,7 %, volatilité 12,5 %, Sharpe 0.46.
  Volatilité : cumulée 1.96, drawdown -39,1 %, volatilité 18,4 %, Sharpe 0.46.
Volatilité minimale
  Sans choc : cumulée 1.84, drawdown -23,1 %, volatilité 11,8 %, Sharpe 0.59.
  Marché : cumulée 1.47, drawdown -25,0 %, volatilité 13,4 %, Sharpe 0.36.
  Taux : cumulée 1.84, drawdown -23,1 %, volatilité 11,8 %, Sharpe 0.42.
  Récession : cumulée 1.45, drawdown -26,0 %, volatilité 12,0 %, Sharpe 0.38.
  Volatilité : cumulée 1.69, drawdown -38,3 %, volatilité 17,6 %, Sharpe 0.39.
Sharpe maxim